# Trabajo Práctico N.º 2 — Informe técnico

## Sistema de detección y clasificación de razas de perros

**Materia:** IA 5.2 — Computer Vision  
**Estudiante:** Lautaro Cena  
**Período:** 1.º cuatrimestre de 2026


## Alcance y criterio del informe

La cátedra proporcionó la infraestructura general del proyecto: estructura de carpetas, API, interfaz Gradio, Docker, scripts de ejecución, persistencia, esquemas y orquestación de los servicios. Por ese motivo, este informe no presenta esos componentes como desarrollo propio.

La documentación se concentra en las funciones que debían completarse, las decisiones experimentales, los resultados obtenidos y las modificaciones adicionales necesarias para integrar el trabajo:

- **Etapa 1:** `extract_embedding`, `search_similar_images` y `predict_breed_from_neighbors`.
- **Etapa 2:** `train_classifier` y `evaluate_classifier`, junto con el entrenamiento y comparación de dos modelos.
- **Etapa 3:** `detect_dogs` y `classify_detected_dog`.

Las evidencias experimentales completas se encuentran en `etapa1_colab.ipynb`, `etapa2_colab.ipynb` y `etapa3_colab.ipynb`.


## 1. Pipeline desarrollado

El trabajo sigue el flujo incremental definido en la consigna:

```text
Imagen
  ├─ Etapa 1: embedding → búsqueda vectorial → vecinos → raza por votación
  ├─ Etapa 2: clasificación supervisada directa
  └─ Etapa 3: YOLO → bounding boxes → crops → clasificación de cada perro
```

### Etapa 1 — Búsqueda por similitud

1. La imagen se carga con OpenCV.
2. Se convierte de BGR a RGB.
3. Una ResNet18 preentrenada, sin su capa final, genera un embedding de 512 dimensiones.
4. El embedding se consulta contra PostgreSQL con `pgvector`.
5. Se recuperan los diez vecinos más cercanos.
6. La raza se estima mediante votación ponderada por similitud.
7. Si el mejor vecino no supera el umbral configurado, se devuelve `unknown`.

### Etapa 2 — Clasificación supervisada

Se entrenaron dos clasificadores sobre las 70 razas:

- ResNet18 con fine-tuning parcial.
- CNN propia entrenada desde cero.

Cada modelo produce una distribución de probabilidad sobre las 70 clases. Los checkpoints conservan los mejores pesos según `valid_loss`, los nombres de las clases, el historial y la época seleccionada.

### Etapa 3 — Detección y clasificación

YOLOv8n localiza perros en una escena completa. Cada bounding box se recorta y se procesa con el mismo preprocesamiento de inferencia utilizado en la Etapa 2. El clasificador devuelve la raza y un score independiente de la confianza de detección de YOLO.


## 2. Dataset y conjuntos de evaluación

Se utilizó **70 Dog Breeds Image Dataset**. Los splits provistos contienen las mismas 70 razas:

| Split | Imágenes | Razas |
|---|---:|---:|
| Train | 7.946 | 70 |
| Valid | 700 | 70 |
| Test | 700 | 70 |

La distribución de `train` presenta un desbalance moderado:

- mínimo: 65 imágenes por raza;
- máximo: 198;
- promedio: 113,5;
- desvío estándar: 24,7;
- 19 de 70 razas tienen menos de 100 imágenes.

El desbalance puede afectar el aprendizaje supervisado y también la representación de cada raza en la base vectorial, aunque en la Etapa 1 el extractor permanece congelado.

### Evaluación externa

Se preparó un conjunto independiente con **26 imágenes de 5 razas**:

- Basset;
- Border Collie;
- Doberman;
- Golden Retriever;
- Poodle.

Este conjunto no reemplaza a `test`, porque solo cubre cinco clases. Se utiliza para observar la generalización frente a imágenes con fondos, iluminación, pose, encuadre y calidad diferentes.

Para la Etapa 3 se emplearon tres imágenes específicas:

| Escenario | Archivo | Perros observados |
|---|---|---:|
| Un perro | `01_one_dog.png` | 1 |
| Múltiples perros | `02_multiple_dogs.jpeg` | 5 |
| Escena compleja | `03_complex_scene.jpg` | 4 |


## 3. Preprocesamiento y reproducibilidad

### Preprocesamiento común

Las imágenes se redimensionan a `224 × 224`, se convierten a tensor y se normalizan con estadísticas de ImageNet:

```text
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

La conversión BGR → RGB es necesaria porque OpenCV y los modelos de `torchvision` usan órdenes de canales diferentes.

### Etapa 1

El indexado usa un preprocesamiento determinista. No se aplica data augmentation porque una misma imagen debe generar una representación estable en la base vectorial.

### Etapa 2

Durante entrenamiento se aplicaron:

- `RandomHorizontalFlip(p=0.5)`;
- rotación de hasta ±15°;
- variaciones de brillo y contraste de ±0,2.

No se aplicaron blur ni ruido sintético. En este problema, la textura del pelaje es informativa y una degradación artificial excesiva podía eliminar rasgos discriminativos.

Validación, test y evaluación externa usan únicamente resize, tensor y normalización.

### Control de calidad

Se verificaron los tres splits mediante PIL y se fijó una resolución mínima de 64 píxeles por lado. No se eliminaron imágenes: todos los archivos pudieron abrirse y superaron el umbral.

También se corrigió una inconsistencia real entre splits:

```text
valid: "American  Spaniel" → "American Spaniel"
```

Luego de la normalización, `train`, `valid` y `test` conservaron exactamente las mismas 70 clases.

### Reproducibilidad

En la notebook de Etapa 2 se fijó la semilla 42 para Python, NumPy y PyTorch. Además, se configuró cuDNN en modo determinista. Esto reduce la variabilidad entre ejecuciones, aunque no garantiza identidad absoluta entre plataformas o versiones.


## 4. Etapa 1 — Implementación y resultados

### 4.1 `extract_embedding`

Se utilizó ResNet18 preentrenada en ImageNet, eliminando la capa `fc`. La salida posterior al average pooling tiene dimensión 512.

El modelo y las transformaciones se cargan una sola vez en `SimilarityService.__init__`. La función:

1. valida el orden de canales;
2. aplica el preprocesamiento;
3. crea un batch de una imagen;
4. ejecuta inferencia con `torch.no_grad()`;
5. aplana la salida y devuelve `list[float]`.

### 4.2 `search_similar_images`

La implementación admite dos backends:

- **PostgreSQL + pgvector:** búsqueda nativa mediante el operador de distancia coseno `<=>`;
- **JSON:** cálculo de similitud contra todos los registros y ordenamiento descendente.

La evaluación final se realizó con PostgreSQL y `pgvector`. Se agregó el cast `%s::vector` en la consulta SQL para evitar que psycopg enviara la consulta como `double precision[]`, tipo incompatible con el operador vectorial.

### 4.3 `predict_breed_from_neighbors`

La raza se decide mediante voto ponderado: cada vecino suma su score a la raza correspondiente. Esta estrategia conserva más información que el voto simple.

Se configuró:

| Parámetro | Valor |
|---|---:|
| Métrica | Cosine similarity |
| Top K | 10 |
| Umbral de similitud | 0,55 |
| Dimensión | 512 |

Si el vecino más cercano no alcanza 0,55, se devuelve `unknown`.

### 4.4 Base vectorial

Se indexaron las 7.946 imágenes de `train`, pertenecientes a 70 razas. El estado final verificado fue:

```text
7.946 registros
7.946 paths únicos
70 razas
embeddings de 512 dimensiones
```

Durante las pruebas se detectó una reindexación parcial repetida. Como cada inserción generaba un UUID nuevo, una misma ruta podía quedar más de una vez. Se depuró la tabla usando `path` como criterio de unicidad para el conjunto ya construido y se dejó desactivada la bandera de indexación completa para evitar reconstrucciones accidentales.

### 4.5 Evaluación

| Conjunto | Imágenes | NDCG@10 medio | Mediana | Rankings perfectos |
|---|---:|---:|---:|---:|
| Test interno | 700 | **0,8770** | 1,0000 | 394 (56,29 %) |
| Externo | 26 | **0,5298** | 0,5997 | 3 (11,54 %) |

La caída externa muestra sensibilidad al cambio de dominio. El tamaño reducido del conjunto externo limita el alcance de esa comparación.

### 4.6 Visualización

Para una muestra de diez razas, los dos primeros componentes de PCA explicaron 24,7 % de la varianza. t-SNE mostró agrupamientos reconocibles, aunque con proximidad entre algunas razas. Estas proyecciones son cualitativas; la métrica principal de recuperación sigue siendo NDCG@10.


## 5. Etapa 2 — Modelos y entrenamiento

### Modelo A: ResNet18 Fine-Tuned

Se reemplazó la capa final por una salida de 70 clases. Se congeló la mayor parte de la red y se dejaron entrenables:

- `layer4`;
- la nueva capa `fc`.

La estrategia conserva las características generales aprendidas en ImageNet y adapta las capas finales al dominio de razas.

### Modelo B: CNN Custom

La CNN propia contiene cuatro bloques convolucionales:

```text
Conv2d → BatchNorm → ReLU → reducción espacial
```

Los canales progresan 32 → 64 → 128 → 256. Al final se aplican `AdaptiveAvgPool2d`, `Flatten`, `Dropout(0.3)` y una capa lineal de 70 salidas.

### Hiperparámetros

| Hiperparámetro | Valor |
|---|---:|
| Batch size | 32 |
| Máximo de épocas | 15 |
| Patience | 4 |
| LR cabeza / CNN | 1e-3 |
| LR `layer4` | 1e-4 |
| Optimizador | Adam |
| Pérdida | CrossEntropyLoss |
| Scheduler | StepLR |
| Step size | 5 |
| Gamma | 0,5 |

El checkpoint se seleccionó por menor `valid_loss`, no por máxima accuracy. Se guardaron el mejor `state_dict`, las clases, el historial, la mejor época y la cantidad de épocas ejecutadas.

El entrenamiento definitivo se realizó en Google Colab con GPU. La ejecución local en una máquina virtual con CPU y memoria limitada resultó excesivamente lenta e inestable. El uso de Colab no cambió el código del proyecto: la notebook clonó la rama `desarrollo`, importó los servicios y descargó los checkpoints al finalizar cada modelo.


## 6. Etapa 2 — Resultados y comparación

### Métricas sobre `test`

Las métricas macro tratan cada raza con el mismo peso.

| Modelo | Accuracy | Precision macro | Recall macro | Specificity macro | F1 macro |
|---|---:|---:|---:|---:|---:|
| ResNet18 Fine-Tuned | **0,9586** | **0,9601** | **0,9586** | **0,9994** | **0,9566** |
| CNN Custom | 0,3714 | 0,3709 | 0,3714 | 0,9909 | 0,3424 |

La specificity es alta en ambos modelos porque se calcula one-vs-rest sobre 70 clases: para cada clase existe una cantidad muy grande de verdaderos negativos. Por eso no debe interpretarse de forma aislada.

### Evolución del entrenamiento

**ResNet18**

- 13 épocas ejecutadas;
- mejor `valid_loss`: 0,1269 en la época 9;
- mejor `valid_acc`: 0,9614 en la época 10;
- accuracy final de train: 0,9946;
- accuracy final de valid: 0,9571.

Luego del mínimo de validación, la pérdida de entrenamiento siguió bajando mientras la de validación empeoró levemente. Esto es compatible con una fase posterior de sobreajuste; el checkpoint guardado evita utilizar esos pesos finales.

**CNN Custom**

- 15 épocas ejecutadas;
- mejor `valid_loss`: 2,3507 en la época 14;
- mejor `valid_acc`: 0,3600 en la época 14;
- accuracy final de train: 0,3200;
- accuracy final de valid: 0,3400.

La CNN no logró aprender representaciones suficientemente discriminativas para 70 razas con el volumen disponible.

### Costo computacional

| Modelo | Parámetros totales | Parámetros entrenables | Inferencia batch 1 |
|---|---:|---:|---:|
| ResNet18 Fine-Tuned | 11.212.422 | 8.429.638 | 2,94 ms |
| CNN Custom | 407.366 | 407.366 | 0,72 ms |

La CNN usa aproximadamente 27,5 veces menos parámetros y fue unas cuatro veces más rápida en el entorno medido, pero la pérdida de calidad fue demasiado grande. Los tiempos son relativos al hardware y backend de esa ejecución.

### Análisis de errores

Las clases con menor recall en ResNet18 fueron:

| Raza | Recall | Confusión principal |
|---|---:|---|
| Malinois | 0,30 | German Sheperd (4) |
| Bulldog | 0,70 | Boston Terrier (2) |
| American Spaniel | 0,80 | Irish Spaniel (2) |
| Beagle | 0,90 | Bluetick (1) |
| Chow | 0,90 | Pomeranian (1) |

En la CNN Custom, Airedale, Borzoi, Corgi, Chihuahua y Chow obtuvieron recall 0 en las diez imágenes de test de cada clase. Los errores fueron más dispersos y muestran que el modelo no aprendió una separación estable.

### Evaluación externa

| Modelo | Accuracy top-1 | Recall macro de las 5 razas | Predicciones fuera de las 5 razas |
|---|---:|---:|---:|
| ResNet18 Fine-Tuned | **0,5385** | **0,5333** | 12/26 |
| CNN Custom | 0,1538 | 0,1400 | 22/26 |

No se reportó una precision macro restringida a las cinco razas porque ambos modelos pueden predecir cualquiera de las 70 clases; excluir las clases ausentes produciría una lectura engañosa.

La evidencia favorece claramente a ResNet18 Fine-Tuned como clasificador final.


## 7. Etapa 3 — Detección y clasificación

### 7.1 `detect_dogs`

La función integrada:

- carga YOLOv8n de forma lazy y lo conserva en caché;
- valida imágenes vacías;
- filtra exclusivamente la clase `dog` de COCO (`class_id=16`);
- aplica el umbral de confianza;
- convierte las coordenadas a píxeles;
- recorta las cajas a los límites de la imagen;
- devuelve `((x1, y1, x2, y2), det_score)`.

En la notebook experimental se utilizaron además:

| Parámetro | Valor |
|---|---:|
| YOLO | `yolov8n.pt` |
| Confidence threshold | 0,25 |
| Image size | 960 |
| NMS IoU | 0,70 |
| Umbral para cajas casi contenidas | 0,90 |

El posprocesamiento de la notebook ordenó las cajas por confianza y eliminó detecciones casi contenidas dentro de otras para evitar contar dos veces un mismo perro.

### 7.2 `classify_detected_dog`

La función:

1. valida el crop;
2. carga el checkpoint activo;
3. reconstruye ResNet18 o la CNN Custom;
4. carga el `state_dict`;
5. conserva el modelo reconstruido en caché;
6. convierte el crop BGR a RGB;
7. aplica el preprocesamiento de inferencia;
8. calcula Softmax;
9. devuelve raza y probabilidad máxima.

### 7.3 Resultados

| Escenario | Perros reales | Detectados | Cobertura |
|---|---:|---:|---:|
| Un perro | 1 | 1 | 100 % |
| Múltiples perros | 5 | 5 | 100 % |
| Escena compleja | 4 | 3 | 75 % |
| **Total** | **10** | **9** | **90 %** |

No hubo detecciones adicionales. El único falso negativo apareció en la escena compleja, donde uno de los perros era más pequeño, distante u ocluido.

Ejemplos de salida observados:

- imagen individual: una detección clasificada como Doberman;
- imagen múltiple: cinco detecciones, con una raza y dos scores por caja;
- escena compleja: tres detecciones con scores de clasificación más bajos y variables.

`det_score` y `breed_score` miden tareas diferentes. Un score alto de clasificación no demuestra que la raza sea correcta.

La evaluación de esta etapa verificó cobertura por conteo y ejecución del pipeline. No se construyó una anotación completa de bounding boxes y razas verdaderas; por lo tanto, no se reportan mAP, IoU medio ni accuracy de raza para estas tres escenas.


## 8. Comparación entre enfoques

### Similitud vs clasificación supervisada

La búsqueda por similitud no requiere entrenamiento específico y es interpretable porque muestra los vecinos que sustentan la decisión. Su rendimiento depende de la representatividad de la base y del dominio de la consulta.

La clasificación supervisada aprende directamente límites de decisión entre 70 clases. ResNet18 Fine-Tuned obtuvo mayor calidad predictiva, pero requiere entrenamiento, checkpoints y control de sobreajuste.

Las métricas no son directamente equivalentes:

- NDCG@10 evalúa la calidad de un ranking;
- accuracy y F1 evalúan clasificación top-1.

### ResNet18 vs CNN propia

La transferencia de aprendizaje fue decisiva. ResNet18 produjo una mejora muy grande en accuracy y F1. La CNN fue más compacta y rápida, pero no alcanzó un nivel adecuado para la aplicación final.

### YOLO + clasificador

Separar localización e identificación permite clasificar perros dentro de escenas completas. YOLO elimina parte del fondo mediante los crops y el clasificador reutiliza el dominio aprendido en imágenes centradas. El rendimiento total queda limitado por ambos componentes: un perro no detectado nunca puede ser clasificado, y una detección correcta puede recibir una raza equivocada.


## 9. Problemas encontrados y soluciones

### Incompatibilidad de tipos en pgvector

**Problema:** psycopg enviaba la lista como `double precision[]`, incompatible con `<=>`.

**Solución:** cast explícito del parámetro a `vector` en la consulta SQL.

### Duplicación durante el indexado

**Problema:** una ejecución interrumpida y luego repetida agregó rutas duplicadas con UUID diferentes.

**Solución aplicada al conjunto final:** deduplicación por `path`, verificación de 7.946 registros únicos y desactivación del indexado completo luego de construir la base.

### Nombres de clases inconsistentes

**Problema:** `ImageFolder` interpretaba espacios dobles como una clase distinta.

**Solución:** normalización de espacios en los nombres y comprobación de igualdad entre splits.

### Preprocesamiento BGR/RGB

**Problema:** OpenCV carga BGR, mientras que los modelos esperan RGB.

**Solución:** conversión explícita antes de embeddings o clasificación.

### Limitaciones de ejecución local

**Problema:** entrenar ambos modelos dentro de VirtualBox con CPU y memoria limitada produjo tiempos de varias horas y presión de RAM/swap.

**Solución:** entrenamiento definitivo en Colab con GPU, descarga inmediata de cada checkpoint y conservación de la ejecución completa en la notebook.

### Pérdida del estado de Colab

**Problema:** una reconexión eliminó variables de Python antes de la descarga final.

**Solución:** celdas de recuperación independientes que reconstruyen configuración y localizan los checkpoints sin volver a entrenar.

### Generalización externa

**Problema:** caída marcada fuera del dataset.

**Solución metodológica:** separar claramente test interno y evaluación externa, reportar el subconjunto evaluado y evitar conclusiones absolutas con 26 imágenes.

### Escenas complejas

**Problema:** cajas duplicadas iniciales y un perro no detectado en la escena compleja.

**Solución:** posprocesamiento de cajas casi contenidas en la notebook y documentación del falso negativo restante, sin forzar un resultado perfecto.


## 10. Modificaciones fuera de las funciones indicadas

Estas modificaciones se realizaron para integrar y ejecutar correctamente las funciones pedidas:

| Archivo / componente | Modificación | Justificación |
|---|---|---|
| `SimilarityService.__init__` | carga de ResNet18, dispositivo y transformaciones | evitar recargar el extractor por imagen |
| `PgVectorEmbeddingStore.search` | cast `%s::vector` | compatibilidad psycopg/pgvector |
| `config.py` y `.env.local.example` | hiperparámetros, rutas y parámetros de evaluación | evitar hardcodear configuración |
| `bootstrap.py` | entrega de `settings` a `ClassifierService` | centralizar hiperparámetros |
| `ClassifierService.__init__` | dispositivo y referencia a `Settings` | entrenamiento configurable |
| Helpers de `ClassifierService` | transformaciones y constructores de modelos | reutilizar exactamente la misma arquitectura en entrenamiento, evaluación e inferencia |
| DataLoaders | workers y `pin_memory` condicionados por dispositivo | reducir presión de memoria en CPU |
| `.gitignore` | exclusión de embeddings JSON y evaluaciones externas | evitar subir artefactos generados |
| Notebooks | detección de entorno, rutas portables, descarga de datos y checkpoints | reproducibilidad local/Colab |

No se presenta como desarrollo propio la API, Gradio, Docker Compose ni la orquestación general provista por la cátedra.


## 11. Alcance verificado y limitaciones

### Limitaciones experimentales

- El conjunto externo de Etapa 2 tiene 26 imágenes y solo 5 de 70 razas.
- Las tres escenas de Etapa 3 no poseen anotaciones completas de raza y bounding boxes.
- Los tiempos de inferencia dependen del hardware.
- La base vectorial se evaluó con ResNet18 baseline; no se compararon distintos extractores mediante NDCG bajo una misma metodología.

### Observación sobre `extract_custom_embedding`

La consigna incluye `extract_custom_embedding(image)` para reutilizar la penúltima capa de los modelos entrenados en la búsqueda por similitud. En la versión del repositorio revisada, `ClassifierService` documenta esa función y la API intenta invocarla, pero el método no aparece implementado en el cuerpo de la clase.

Por esta razón, este informe no atribuye como completada la selección dinámica de ResNet18 Fine-Tuned o CNN Custom como extractores de embeddings. Los resultados de Etapa 1 corresponden al extractor baseline de 512 dimensiones.

Además, la CNN Custom produce una representación intermedia de 256 dimensiones, mientras que la tabla pgvector final fue creada para vectores de 512. Completar esa integración requiere definir una estrategia explícita —por ejemplo, modificar la arquitectura y reentrenar, usar una proyección entrenada o mantener índices separados por modelo—; no sería correcto resolverlo mediante padding arbitrario y presentar esa salida como equivalente.

Esta observación debe resolverse antes de considerar cumplido el requisito de selección dinámica de extractor indicado en la consigna.


## 12. Conclusiones

La Etapa 1 demostró que un extractor general preentrenado puede producir rankings útiles sin fine-tuning: NDCG@10 fue 0,8770 en test y 0,5298 en el conjunto externo.

En la Etapa 2, ResNet18 Fine-Tuned fue claramente superior a la CNN Custom:

- accuracy: 0,9586 frente a 0,3714;
- F1 macro: 0,9566 frente a 0,3424.

La CNN redujo mucho el tamaño y el tiempo de inferencia, pero el costo en calidad fue demasiado alto. Por eso se seleccionó ResNet18 Fine-Tuned para clasificar los crops de la Etapa 3.

El pipeline de detección cubrió 9 de 10 perros visibles en los tres escenarios. Funcionó de manera completa en las imágenes simple y múltiple, y mostró una limitación real en la escena compleja.

Los experimentos también evidencian un cambio de dominio importante: tanto la recuperación como la clasificación pierden rendimiento sobre imágenes externas. El resultado final respalda el uso de transferencia de aprendizaje y la separación entre detección y clasificación, pero también muestra que una evaluación interna alta no garantiza el mismo comportamiento en condiciones no controladas.


## Anexo — Trazabilidad

| Requisito | Implementación / evidencia |
|---|---|
| Embeddings preentrenados | `SimilarityService.extract_embedding` |
| Base vectorial | PostgreSQL + `pgvector`, 7.946 registros |
| Top-10 y raza por vecinos | `search_similar_images` y `predict_breed_from_neighbors` |
| NDCG@10 | `etapa1_colab.ipynb` |
| Fine-tuning ResNet | `ClassifierService.train_classifier` y `etapa2_colab.ipynb` |
| CNN propia | `_build_cnn_custom` y entrenamiento en Etapa 2 |
| Métricas obligatorias | `evaluate_classifier` y tablas de Etapa 2 |
| Matrices y curvas | salidas guardadas en `etapa2_colab.ipynb` |
| YOLO preentrenado | `DetectionService.detect_dogs` |
| Clasificación por crop | `classify_detected_dog` |
| Un perro / múltiples / escena compleja | `etapa3_colab.ipynb` |
| Configuración sin valores dentro de las funciones | `config.py`, `.env` y `.env.local.example` |
| Modificaciones adicionales justificadas | sección 10 de este informe |
